# Real data analysis project in `duckdb`

In this notebook, we will analyze a big dataset which would be
problematic in pure Python.

We want to take a look at the [arXiv](https://arxiv.org) data. arXiv is (one of the)
biggest library of preprint papers which is available on the 
Internet. Assembling the data would be complicated (and interesting),
but this has already been done by the Cornell University. The
resulting dataset is published on [Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv).

You will see that the dataset has certain *challenges* like nested fields,
time and date in different formats and string fields which acutally contain
lists. Moreover, the dataset is quite large. Therefore, a ZIP file with
JSON data is provided in Kaggle.

## Setup

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    os.system("pip install -U duckdb polars")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

## Download data

The data with all arXiv abstracts is available on [Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv).
Fortunately, it is also possible to download the dataset without logging into Kaggle.

As download can take a while depending on the speed of your Internet connection,
the download is organized in chunks which allow a progress display:

In [ ]:
from tqdm.auto import tqdm
import requests

response = requests.get("https://www.kaggle.com/api/v1/datasets/download/Cornell-University/arxiv", stream=True)
total_size = int(response.headers.get("content-length", 0))
block_size = 1024

with tqdm(total=total_size, unit="B", unit_scale=True) as progress_bar:
    with open("arxiv.zip", "wb") as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)

if total_size != 0 and progress_bar.n != total_size:
    print("Could not download file")

As explained above, the data is provided in `ZIP` format.
To avoid unpacking, we can use the `zipfs` extension in 
`duckdb` and access the data in the `ZIP` file directly.

The extension needs to be installed first, then it can be
loaded. Even if you have installed it before, it does not
hurt to reinstall (and it is necessary if you use Google
Colab and have a new session):

## Ingest data

In [ ]:
import duckdb
duckdb.sql("INSTALL zipfs FROM community; LOAD zipfs") 

If the extension does not work for you, you can 
extract he files manually or use the following code
to extract the files (adjust the names in following the `duckdb`
statements appropriately):
```python
from zipfile import ZipFile
with ZipFile("arxiv.zip", 'r') as zip_ref:
    zip_ref.extractall(".")
```

First convert the data to `parquet` format so we have fast access. We will use
the current timestamp in the converted `parquet` file. There are a 
few reasons for that:
* New versions of `arxiv.zip` become available every few days.
* We don't want to overwrite an already existing converted `parquet` file.
* If we use more sophisticated analysis (like NLP or LLMs) on the data, we
  would like to restrict that on the new data (incremental updates). 

In [ ]:
from datetime import datetime
today = datetime.now().strftime('%Y%m%d')
today

Great, now we have a timestamp and can convert the data.

Converting the data to `parquet` format in the first step might
look a bit *exaggerated*, but as it is so much faster to access
the data, it is often a good idea. That's especially true for
data in `ZIP` files, as it needs to be extracted first.

Therefore, let's convert the data to a timestamped `parquet` file:

In [ ]:
%%time
# remove the zip://arxiv.zip/ if you unpacked the file above:
duckdb.sql(f"COPY (SELECT * FROM 'zip://arxiv.zip/arxiv-metadata-oai-snapshot.json') TO 'arxiv-{today}.parquet'")

This takes considerable time, especially if disks are slow (or on Colab).
Depending on your CPU configuration, you will see that `duckdb` is
inherently multi-threaded.

After the conversion, we can take a look at the data (sounds counter-intuitive,
but the conversion does not lose any data). First check the number of articles:

## General Structure and Volume

In [ ]:
duckdb.sql(f"SELECT count(*) FROM 'arxiv-{today}.parquet'").pl()

Wow, that's really many articles. A `DataFrame` with so many articles would
be possible to handle in Python, but that depends heavily on the column
structure. 

As the `parquet` file is stored in columnar format, selecting all rows is
very expensive. For the following statements, we use a restriction of five
rows to keep the statements fast.

Let's check the structure:

In [ ]:
duckdb.sql(f"FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

14 columns is not a lot, but the `abstract` column can be quite long.

We are more interested in the details of the columns, therefore we use
`duckdb`'s `DESCRIBE` command:

In [ ]:
duckdb.sql(f"DESCRIBE FROM 'arxiv-{today}.parquet'").pl().show(limit=15, fmt_str_lengths=100)

There are columns which have complex structure:
* `authors_parsed` contains the names (last, first, middle) of the authors. Currently, that's not too
  interesting for us.
* `version` is much more interesting, as it contains the publication
  history of the articles. We are interested in the first date of publication.
  How can we find that? For this, we first have to `unnest` the column:

## Versions and Date

In [ ]:
duckdb.sql(f"SELECT id, unnest(versions) FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

This has worked great! 

Note however that for each original article we now have as many copies
as there are version of that article. 

Note additionally that the `unnest` is not complete: there is still a
substructure in the column (containing the `version` and the `created` field).

Fortunately, `duckdb` can also solve this by *recursively* unpacking 
the field:

In [ ]:
duckdb.sql(f"SELECT id, unnest(versions, recursive := true) FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

Excellent, but still we have to cope with the multiplicity.

To solve this, note that each article needs to have a least a `version` `v1`.
Let's try to select that:

In [ ]:
duckdb.sql(f"SELECT id, unnest(versions, recursive := true) FROM 'arxiv-{today}.parquet' WHERE version='v1' LIMIT 5").pl()

Unfortunately, this does not work as expected. We cannot reference a column
`version` that is not part of the `SELECT`.

We could create a temporary table, but this would involve copying a lot of
data (which is expensive in the columnar format). Instead, we will work with
a CTE as an *alias* which should give us the right columns. We `EXCLUDE`
the original `versions` field as we do not need it anymore:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT * EXCLUDE(versions) FROM publications WHERE version='v1' LIMIT 5").pl()

This statement takes a while. The reason is that all columns need
to be read which is not necessary right now. Let's use only `id`
and `created`:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, created FROM publications WHERE version='v1' LIMIT 5").pl()

That's much faster!

Note that each article is only present once which you
can see by the `id` fields being unique.

In the next step, we will convert the `created` field
to a real timestamp (as it is a `str` right now).
`duckdb` has powerful functions for that like `strptime`:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, created, strptime(created, '%a, %d %b %Y %H:%M:%S %Z') FROM publications WHERE version='v1' LIMIT 5").pl()

## Categories

Let's consider the next field `categories`. Even though this is a `VARCHAR`
field, in reality it contains a list of the categories the articles has been
assigned to:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, categories FROM publications WHERE version='v1' LIMIT 5").pl()

As this is not a list (why?), we cannot use `unnest` here. `duckdb` also
has powerful string functions which can split strings and create a list:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, string_split(categories, ' ') AS category_list FROM publications WHERE version='v1' LIMIT 5").pl()

Very good, now we have a list of categories and can use filtering functions
in `duckdb` for our further analysis!

## Authors

The final complex field is `authors_parsed`:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, authors_parsed FROM publications WHERE version='v1' LIMIT 5").pl()

If you look closely, this is already in the format which we would like to have.
It is a list of lists, the outer list is for the authors, the inner list
contains there names (last name, first name, middle name). We do not need to change this.

## Convert the structure to a new `parquet` file

With the `version` `v1` of the articles, the publication date, and
the list of categories we now have a "normalized" version of the data
available.

For further data analysis, it is a good idea to save that structure
to a dedicated `parquet` file. This might be a time-intensive task,
therefore check our conversion first with a smaller subset:

In [ ]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT * EXCLUDE versions, \
                    strptime(created, '%a, %d %b %Y %H:%M:%S %Z') AS publication_date, \
                    string_split(categories, ' ') AS category_list \
                    FROM publications WHERE version='v1' LIMIT 5").pl()

If we wanted, we could remove `version` and `created` using another CTE,
but it might be good to keep it for further debugging. Therefore, let's
perform the conversion and persist it in `parquet`. Note that the statement
is almost identical to the one which you see above except for the `LIMIT 5`
and everything in a `COPY` "container":

In [ ]:
%%time
duckdb.sql(f"COPY (WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
                   SELECT * EXCLUDE versions, \
                          strptime(created, '%a, %d %b %Y %H:%M:%S %Z') AS publication_date, \
                          string_split(categories, ' ') AS category_list \
                          FROM publications WHERE version='v1') TO 'arxiv-normalized-{today}.parquet'")

That was fast, but a lot of CPU time was used by `duckdb`
(probably for packing the data). We could also have used a higher
compression level, but that takes even longer (don't be fooled by
the progress bar). Don't run this during the course, it takes
*really long* (8min on my computer):

In [ ]:
%%time
Please run it later
duckdb.sql(f"COPY (WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
                   SELECT * EXCLUDE versions, \
                          strptime(created, '%a, %d %b %Y %H:%M:%S %Z') AS publication_date, \
                          string_split(categories, ' ') AS category_list \
                          FROM publications WHERE version='v1') TO 'arxiv-normalized-l22-{today}.parquet'\
                            (FORMAT 'PARQUET', CODEC  'zstd', COMPRESSION_LEVEL 22, ROW_GROUP_SIZE 100000)")

Space is definitely saved with this compression, the file is not even half the size!

Therefore, it *might* be useful to save data with such high compression factors if
you need a long term storage or if your disk is slow (as it increases read performance):

In [ ]:
!ls -l arxiv-normalized-*.parquet

The structure of the files is of course identical, therefore we
can `DESCRIBE` any of the *normalized* files:

In [ ]:
duckdb.sql(f"DESCRIBE FROM 'arxiv-normalized-{today}.parquet'").pl().show(limit=100, fmt_str_lengths=100)

# Data analysis

Until now, we have just performed data transformations.
Although that might look boring, it is often one of the
most challenging parts of data analysis.

Without having a fast, powerful tool like `duckdb`, this
would take much longer. Esprecially `unnest` and the 
`date` functions have been very useful for us. 

As we have the data now in the correct format, we can turn
to the actual analysis.

## Find duplicates

Taking a look at the structure of the data above, the `id` field
should be unique. But is it really? Using aggregation functions this
is quite easy to check:

In [ ]:
duckdb.sql(f"SELECT id, COUNT(*) AS count FROM 'arxiv-normalized-{today}.parquet'\
                GROUP BY ALL HAVING COUNT>1 ORDER BY count DESC").pl().show(limit=10)

Obviously, there are quite a number of duplicates. How many exactly? Use a CTE to find out:

In [ ]:
duckdb.sql(f"WITH duplicates AS (SELECT id, COUNT(*) AS count FROM 'arxiv-normalized-{today}.parquet'\
                GROUP BY ALL HAVING COUNT>1 ORDER BY count DESC) \
             SELECT COUNT(count), SUM(count) FROM duplicates").pl()

79 duplicates is not that many for 5 million articles. A few appear
more than twice.

Let's check the articles which have most duplicates in the `id` whether
`title` etc. are also the same:

In [ ]:
duckdb.sql(f"SELECT * FROM 'arxiv-normalized-{today}.parquet' WHERE id IN ('math-ph/0605055', 'math-ph/0512019') ORDER BY id").pl()

Indeed, these articles seem to be identical. This could have several reasons:
* Problems with the submission of the article.
* Problems in creating the dataset.
* Problems in our `unnest` (like e.g. having `v1` twice in the `version` field).

We can only check the last problem by taking a look at the original data.
Fortunately, this is also available in `parquet` format:

In [ ]:
duckdb.sql(f"SELECT * FROM 'arxiv-{today}.parquet' WHERE id IN ('math-ph/0605055', 'math-ph/0512019') ORDER BY id").pl()

At least, we can rule out the last problem. 

Taking a look at https://arxiv.org/list/math-ph/2006-05?show=2000, it seems
that the article is present in the list only once. So maybe it is a problem
originating in the dataset assembly.

As these are so few articles, we will ignore the problem in the further analysis.

# Author analysis

Let's try to find the most common authors:

In [ ]:
duckdb.sql(f"WITH article_authors AS (SELECT *, unnest(authors_parsed) AS authors FROM 'arxiv-normalized-{today}.parquet')\
                SELECT authors, COUNT(*) AS count FROM article_authors GROUP BY ALL ORDER BY count DESC LIMIT 50").pl().show(limit=50)

Not very surprisingly, the most prominent authors are *collaboration*.
Especially in experimental particle physics, many, many scientists
participate in experiments (like in CERN) and publish their results
together.

# Computer Science Development

Most of us will be somehow associated with computer science. arXiv also has
a lot of articles from that field. These articles have `cs.*` in their `category_list`.

## Data Filtering

In the first step, we will filter these articles.
    
This sounds simple at first, but we have to perform a trick here. `duckdb` offers a function
called `list_filter`. We can pass a `lambda` function to that list and evaluate that. This
filters all categories which start with `cs.*`:

In [ ]:
duckdb.sql(f"SELECT id, list_filter(category_list, lambda c: c LIKE 'cs.%') AS cs_list FROM 'arxiv-normalized-{today}.parquet'").pl()

In the next step, we only select articles where this `cs_list` has a length > 0:

In [ ]:
duckdb.sql(f"WITH cs_filter AS (SELECT *, list_filter(category_list, lambda c: c LIKE 'cs.%') AS cs_list \
                                       FROM 'arxiv-normalized-{today}.parquet')\
             SELECT id, cs_list FROM cs_filter WHERE length(cs_list)>0").pl()

That looks good, save this to a `parquet` which makes analysis easier for us.
Of course, we could also use a temporary table but if we need the data again
later, we have to filter again:

In [ ]:
%%time
duckdb.sql(f"COPY (WITH cs_filter AS (SELECT *, list_filter(category_list, lambda c: c LIKE 'cs.%') AS cs_list \
                                             FROM 'arxiv-normalized-{today}.parquet')\
                   SELECT * FROM cs_filter WHERE length(cs_list)>0) TO 'arxiv-normalized-cs-{today}.parquet'")

## Time development

How has the volume of computer science papers increased over time? 
We first start with a naive approach by counting all papers which
are in the corresponding subset.

In [ ]:
duckdb.sql(f"SELECT date_trunc('month', publication_date) AS date, COUNT(*) as count \
                    FROM 'arxiv-normalized-cs-{today}.parquet' GROUP BY ALL ORDER BY date").pl()

Looks good, now plot it!

In [ ]:
duckdb.sql(f"SELECT date_trunc('month', publication_date) AS date, COUNT(*) as count \
                    FROM 'arxiv-normalized-cs-{today}.parquet' GROUP BY ALL ORDER BY date").pl().plot.line(x='date', y='count').\
    properties(width=800, title="Total CS articles")

Wow, what an increase!

However, that might also be due to the total number of articles increasing
as arXiv is getting more and more popular. Let's see!

In [ ]:
duckdb.sql(f"SELECT date_trunc('month', publication_date) AS date, COUNT(*) as count \
                    FROM 'arxiv-normalized-{today}.parquet' GROUP BY ALL ORDER BY date").pl().plot.line(x='date', y='count').\
    properties(width=800, title="Total articles")

Looks very similar. With these diagrams, it is difficult to tell whether
the fraction of CS articles has increased.

If we want to calculate that fraction directly in `duckdb`, we have to go back to our inital
file. Alternatively, we could also work with both dataframes from above and
use Python to calculate the fraction.

In [ ]:
duckdb.sql(f"WITH cs_filter AS (SELECT *, list_filter(category_list, lambda c: c LIKE 'cs.%') AS cs_list \
                                       FROM 'arxiv-normalized-{today}.parquet')\
             SELECT date_trunc('month', publication_date) AS date, COUNT(*) as total_count,\
                    COUNT(*) FILTER (length(cs_list)>0) AS cs_count, cs_count/total_count AS cs_fraction \
                    FROM cs_filter GROUP BY ALL ORDER BY date").pl()

Looks good, let's plot it!

In [ ]:
duckdb.sql(f"WITH cs_filter AS (SELECT *, list_filter(category_list, lambda c: c LIKE 'cs.%') AS cs_list \
                                       FROM 'arxiv-normalized-{today}.parquet')\
             SELECT date_trunc('month', publication_date) AS date, COUNT(*) as total_count,\
                    COUNT(*) FILTER (length(cs_list)>0) AS cs_count, cs_count/total_count AS cs_fraction \
                    FROM cs_filter GROUP BY ALL ORDER BY date").pl().plot.line(x="date", y="cs_fraction").\
    properties(width=800, title="Fraction of CS articles")

This is a much more insightful diagram. As you can see, the fraction of CS
articles has increased *tremendously*. With some outliers at the very beginning
(i.e. before 1995) the fraction has increased continuously from less than 3%
in the 1992 to more than 50% after 2024.

## Changing Focus in Computer Science

The computer science field is quite broad, i.e. there are many different
subfields. Let's check the most popular of these subfields. For this, we use
the filtered dataset.

First, we `unnest` the `category_list` in a CTE, then we filter only the CS
categories. This is necessary, as some articles appear e.g. both in CS and in
MATH:

In [ ]:
duckdb.sql(f"WITH cs_category AS (SELECT id, publication_date, unnest(category_list) AS category \
                                         FROM 'arxiv-normalized-cs-{today}.parquet') \
                 SELECT * FROM cs_category WHERE category LIKE 'cs.%'").pl()

Now, we want to count how many articles appear in which year in which CS category:

In [ ]:
duckdb.sql(f"WITH cs_category AS (SELECT id, publication_date, unnest(category_list) AS category \
                                         FROM 'arxiv-normalized-cs-{today}.parquet') \
                 SELECT date_trunc('year', publication_date) AS year, category, COUNT(*) AS count\
                        FROM cs_category WHERE category LIKE 'cs.%' GROUP BY ALL ORDER BY year").pl()

We could now use `PIVOT` directly in `duckdb`, however both `polars` and `pandas`
also have powerful `pivot` functions. Both work well, but it turns out that `pandas` is a bit
easier for later conversion to a *heatmap*.

In [ ]:
# polars version
ydf = duckdb.sql(f"WITH cs_category AS (SELECT id, publication_date, unnest(category_list) AS category \
                                               FROM 'arxiv-normalized-cs-{today}.parquet') \
                       SELECT date_trunc('year', publication_date) AS year, category, COUNT(*) AS count\
                              FROM cs_category WHERE category LIKE 'cs.%' GROUP BY ALL ORDER BY year").pl().\
             pivot("year", index="category", values="count")
ydf

In [ ]:
# pandas version
ydf = duckdb.sql(f"WITH cs_category AS (SELECT id, publication_date, unnest(category_list) AS category \
                                               FROM 'arxiv-normalized-cs-{today}.parquet') \
                       SELECT date_trunc('year', publication_date) AS year, category, COUNT(*) AS count\
                              FROM cs_category WHERE category LIKE 'cs.%' GROUP BY ALL ORDER BY year").df().\
             pivot(columns="year", index="category", values="count")
ydf

There are too many numbers in this table. If you want a qualitative visualization, you can use a *heatmap*.
Seaborn has a very nice function for creating these heatmaps. Before passing the `DataFrame`, we replace the
`NaN` values with zero (which is justified as there were no articles published in the category/year combination):

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 10))
sns.heatmap(ydf.fillna(0))

This diagram is still a bit misleading, though. The total volume has increased
year over year. Even if `cs.AI` (Artifical Intelligence), `cs.CV` (Computer
Vision and Pattern Recognition), `cs.CL` (Computation and Language), and 
cs.LG (Machine Learning) appear brightest at
the end, the focus might have been different in other years and is hidden
due to the low volume of articles.

To compensate for this, we have to normalize with respect to the years.

In [ ]:
nydf = ydf / ydf.sum()

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(nydf.fillna(0))

Unfortunately, the outliers in the 1990s are now dominating. Let's fix this in a final step:

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(nydf[[c for c in nydf.columns if c.year > 1999]].fillna(0))

If you want a more quantitative comparison of certain key categories,
you can use a line diagram with the same data. As the `index` will become
the `x` axis, we need to transpose the data and skip the first ten 
entries (from the 1990s):

In [ ]:
nydf.T[10:][["cs.AI", "cs.CL", "cs.CV", "cs.IT", "cs.LG"]].plot(figsize=(16,9))

As you can now see in this quantitative diagram, `cs.IT` had its high in 2005 to 2015,
dropping sharply afterwards. `cs.LG` became popular after 2016 (because of GPUs and
Deep Learning) whereas `cs.AI` has been steadily increasing since 2019, probably due
to the advances in language models because of the transformer architecture.